### Setup

In [1]:
import importlib
import polars as pl

pl.Config.set_tbl_rows(25)

from pathlib import Path

REPO_DIR = Path('.').resolve().parent
DEV_DATA_DIR = REPO_DIR / 'trio-dev-data'

import sys
sys.path.insert(0, str(REPO_DIR / 'src'))
sys.path.insert(0, str(REPO_DIR / 'src' / 'util'))

In [2]:
KID_ID = 'NA12878'
DAD_ID = 'NA12891'
MOM_ID = 'NA12892'

VCF_TRIO_PHASED = str(DEV_DATA_DIR / 'output' / 'trio-phasing' / 'CEPH-1463.joint.GRCh38.deepvariant.glnexus.phased.vcf.gz')

def blocks_tsv_path(uid):
    return str(DEV_DATA_DIR / 'output' / 'trio-phasing' / f'CEPH-1463.joint.GRCh38.deepvariant.glnexus.phased.{uid}.blocks.tsv')

BLOCKS_TSV_KID = blocks_tsv_path(KID_ID)
BLOCKS_TSV_DAD = blocks_tsv_path(DAD_ID)
BLOCKS_TSV_MOM = blocks_tsv_path(MOM_ID)

In [3]:
METH_BASE_DIR = DEV_DATA_DIR / 'output' / 'dna-methylation'

def meth_bed_paths(uid):
    """Return (count_hap1, count_hap2, model_hap1, model_hap2) paths for a sample."""
    count_dir = METH_BASE_DIR / 'CEPH1463.GRCh38.hifi.count.trio-phased'
    model_dir = METH_BASE_DIR / 'CEPH1463.GRCh38.hifi.model.trio-phased'
    return (
        str(count_dir / f'{uid}.GRCh38.haplotagged.hap1.bed.gz'),
        str(count_dir / f'{uid}.GRCh38.haplotagged.hap2.bed.gz'),
        str(model_dir / f'{uid}.GRCh38.haplotagged.hap1.bed.gz'),
        str(model_dir / f'{uid}.GRCh38.haplotagged.hap2.bed.gz'),
    )

KID_METH = meth_bed_paths(KID_ID)
DAD_METH = meth_bed_paths(DAD_ID)
MOM_METH = meth_bed_paths(MOM_ID)

### Read per-sample het SNV alleles and phase block IDs from the WhatsHap trio-phased VCF

In [ ]:
import phasing_trio
importlib.reload(phasing_trio)
from phasing_trio import get_pedmec_phasing

PHASING = get_pedmec_phasing(VCF_TRIO_PHASED, KID_ID, DAD_ID, MOM_ID)
DF_KID = PHASING[KID_ID]
DF_DAD = PHASING[DAD_ID]
DF_MOM = PHASING[MOM_ID]

print(f'Kid het SNVs: {len(DF_KID)}')
print(f'Dad het SNVs: {len(DF_DAD)}')
print(f'Mom het SNVs: {len(DF_MOM)}')

Kid het SNVs: 690
Dad het SNVs: 1059
Mom het SNVs: 801


In [10]:
DF_KID

chrom,start,end,REF,ALT,phase_block_id,allele_hap1,allele_hap2
str,i64,i64,str,str,str,str,str
"""chr1""",1000334,1000335,"""C""","""T""","""1000335""","""1""","""0"""
"""chr1""",1000730,1000731,"""C""","""T""","""1000335""","""0""","""1"""
"""chr1""",1002744,1002745,"""G""","""A""","""1000335""","""1""","""0"""
"""chr1""",1004624,1004625,"""A""","""G""","""1000335""","""1""","""0"""
"""chr1""",1004715,1004716,"""C""","""T""","""1000335""","""0""","""1"""
"""chr1""",1005903,1005904,"""C""","""T""","""1000335""","""1""","""0"""
"""chr1""",1005953,1005954,"""G""","""A""","""1000335""","""1""","""0"""
"""chr1""",1006158,1006159,"""C""","""T""","""1000335""","""1""","""0"""
"""chr1""",1008306,1008307,"""G""","""C""","""1000335""","""1""","""0"""


In [7]:
DF_DAD

chrom,start,end,REF,ALT,phase_block_id,allele_hap1,allele_hap2
str,i64,i64,str,str,str,str,str
"""chr1""",1000334,1000335,"""C""","""T""","""1000335""","""1""","""0"""
"""chr1""",1000730,1000731,"""C""","""T""","""1000335""","""0""","""1"""
"""chr1""",1002744,1002745,"""G""","""A""","""1000335""","""1""","""0"""
"""chr1""",1004624,1004625,"""A""","""G""","""1000335""","""1""","""0"""
"""chr1""",1004715,1004716,"""C""","""T""","""1000335""","""0""","""1"""
"""chr1""",1004822,1004823,"""G""","""A""","""1000335""","""0""","""1"""
"""chr1""",1005903,1005904,"""C""","""T""","""1000335""","""1""","""0"""
"""chr1""",1005953,1005954,"""G""","""A""","""1000335""","""1""","""0"""
"""chr1""",1006158,1006159,"""C""","""T""","""1000335""","""1""","""0"""


In [8]:
DF_MOM

chrom,start,end,REF,ALT,phase_block_id,allele_hap1,allele_hap2
str,i64,i64,str,str,str,str,str
"""chr1""",1000334,1000335,"""C""","""T""","""1000335""","""0""","""1"""
"""chr1""",1000730,1000731,"""C""","""T""","""1000335""","""1""","""0"""
"""chr1""",1002744,1002745,"""G""","""A""","""1000335""","""0""","""1"""
"""chr1""",1004624,1004625,"""A""","""G""","""1000335""","""0""","""1"""
"""chr1""",1004715,1004716,"""C""","""T""","""1000335""","""1""","""0"""
"""chr1""",1005903,1005904,"""C""","""T""","""1000335""","""0""","""1"""
"""chr1""",1005953,1005954,"""G""","""A""","""1000335""","""0""","""1"""
"""chr1""",1006158,1006159,"""C""","""T""","""1000335""","""0""","""1"""
"""chr1""",1008306,1008307,"""G""","""C""","""1000335""","""0""","""1"""


### Get phase blocks for each individual (pre-computed by `run-whatshap.sh`)

In [ ]:
import phasing_trio
importlib.reload(phasing_trio)
from phasing_trio import get_phase_blocks

DF_BLOCKS_KID = get_phase_blocks(BLOCKS_TSV_KID, KID_ID)
DF_BLOCKS_DAD = get_phase_blocks(BLOCKS_TSV_DAD, DAD_ID)
DF_BLOCKS_MOM = get_phase_blocks(BLOCKS_TSV_MOM, MOM_ID)

print(f'Kid phase blocks: {len(DF_BLOCKS_KID)}')
print(f'Dad phase blocks: {len(DF_BLOCKS_DAD)}')
print(f'Mom phase blocks: {len(DF_BLOCKS_MOM)}')

Kid phase blocks: 1
Dad phase blocks: 1
Mom phase blocks: 1


In [12]:
DF_BLOCKS_KID

chrom,start,end,phase_block_id,num_variants
str,i64,i64,str,i64
"""chr1""",1000334,1999756,"""1000335""",802


In [13]:
DF_BLOCKS_DAD

chrom,start,end,phase_block_id,num_variants
str,i64,i64,str,i64
"""chr1""",1000334,1999860,"""1000335""",1231


In [14]:
DF_BLOCKS_MOM

chrom,start,end,phase_block_id,num_variants
str,i64,i64,str,i64
"""chr1""",1000334,1993898,"""1000335""",949


### Construct a Hap Map, consisting of intervals ("hap-map blocks") in which haplotypes in the kid are mapped to haplotypes in the parents

Labeling convention:
- **A** = dad's hap1, **B** = dad's hap2 (fixed)
- **C** = mom's hap1, **D** = mom's hap2 (fixed)
- Kid's hap1 (paternal) matches A or B per block; kid's hap2 (maternal) matches C or D per block

Note that A, B, C, D are only defined within hap-map blocks, not chromosome-wide, as is the case for the pedigree-wise workflow

In [17]:
import hap_map_trio
importlib.reload(hap_map_trio)
from hap_map_trio import get_hap_map

get_hap_map(
    DF_KID, DF_DAD, DF_MOM,
    DF_BLOCKS_KID, DF_BLOCKS_DAD, DF_BLOCKS_MOM,
)


chrom,start,end,REF,ALT,phase_block_id,allele_hap1,allele_hap2,chrom_hap_map_block,start_hap_map_block,end_hap_map_block
str,i64,i64,str,str,str,str,str,str,i64,i64
"""chr1""",1000334,1000335,"""C""","""T""","""1000335""","""1""","""0""","""chr1""",1000334,1993898
"""chr1""",1000730,1000731,"""C""","""T""","""1000335""","""0""","""1""","""chr1""",1000334,1993898
"""chr1""",1002744,1002745,"""G""","""A""","""1000335""","""1""","""0""","""chr1""",1000334,1993898
"""chr1""",1004624,1004625,"""A""","""G""","""1000335""","""1""","""0""","""chr1""",1000334,1993898
"""chr1""",1004715,1004716,"""C""","""T""","""1000335""","""0""","""1""","""chr1""",1000334,1993898
"""chr1""",1005903,1005904,"""C""","""T""","""1000335""","""1""","""0""","""chr1""",1000334,1993898
"""chr1""",1005953,1005954,"""G""","""A""","""1000335""","""1""","""0""","""chr1""",1000334,1993898
"""chr1""",1006158,1006159,"""C""","""T""","""1000335""","""1""","""0""","""chr1""",1000334,1993898
"""chr1""",1008306,1008307,"""G""","""C""","""1000335""","""1""","""0""","""chr1""",1000334,1993898


In [ ]:
import hap_map_trio
importlib.reload(hap_map_trio)
from hap_map_trio import get_hap_map

DF_HAP_MAP = get_hap_map(
    DF_KID, DF_DAD, DF_MOM,
    DF_BLOCKS_KID, DF_BLOCKS_DAD, DF_BLOCKS_MOM,
)
print(f'Hap map blocks: {len(DF_HAP_MAP)}')
DF_HAP_MAP

Hap map blocks: 1


chrom,start,end,paternal_haplotype,maternal_haplotype,paternal_concordance,maternal_concordance,num_het_SNVs_pat,num_het_SNVs_mat
str,i64,i64,str,str,f64,f64,i64,i64
"""chr1""",1000334,1993898,"""A""","""C""",1.0,1.0,1,1


In [ ]:
print('Paternal haplotype value counts:')
print(DF_HAP_MAP['paternal_haplotype'].value_counts())
print()
print('Maternal haplotype value counts:')
print(DF_HAP_MAP['maternal_haplotype'].value_counts())

In [ ]:
print('Blocks with paternal concordance < 1.0:')
DF_HAP_MAP.filter(pl.col('paternal_concordance') < 1.0)

In [ ]:
print('Blocks with maternal concordance < 1.0:')
DF_HAP_MAP.filter(pl.col('maternal_concordance') < 1.0)

### Phase kid methylation to A, B, C, D 

This step requires methylation BED files from `aligned_bam_to_cpg_scores` (run on the WhatsHap-haplotagged BAMs).

In [ ]:
import os
for label, paths in [('Kid', KID_METH), ('Dad', DAD_METH), ('Mom', MOM_METH)]:
    for p in paths:
        exists = os.path.exists(p)
        print(f'{label}: {Path(p).name} -> {"EXISTS" if exists else "MISSING"}')

In [ ]:
import get_meth_hap1_hap2
importlib.reload(get_meth_hap1_hap2)
from get_meth_hap1_hap2 import read_meth_hap1_hap2

DF_METH_COUNT_HAP1_HAP2_KID = read_meth_hap1_hap2(
    pb_cpg_tool_mode='count',
    bed_hap1=KID_METH[0],
    bed_hap2=KID_METH[1],
)
print(f'Kid count-based meth: {len(DF_METH_COUNT_HAP1_HAP2_KID)} rows')
DF_METH_COUNT_HAP1_HAP2_KID

In [ ]:
import phase_meth_to_parent_haps
importlib.reload(phase_meth_to_parent_haps)
from phase_meth_to_parent_haps import phase_meth_to_haplotypes

DF_METH_COUNT_PHASED_KID = phase_meth_to_haplotypes(DF_METH_COUNT_HAP1_HAP2_KID, DF_HAP_MAP, 'kid')
print(f'Kid phased meth: {len(DF_METH_COUNT_PHASED_KID)} rows')
DF_METH_COUNT_PHASED_KID.filter(pl.col('start_hap_map_block').is_not_null()).head(10)

In [ ]:
DF_METH_COUNT_HAP1_HAP2_DAD = read_meth_hap1_hap2(
    pb_cpg_tool_mode='count',
    bed_hap1=DAD_METH[0],
    bed_hap2=DAD_METH[1],
)
DF_METH_COUNT_PHASED_DAD = phase_meth_to_haplotypes(DF_METH_COUNT_HAP1_HAP2_DAD, DF_HAP_MAP, 'dad')
print(f'Dad phased meth: {len(DF_METH_COUNT_PHASED_DAD)} rows')
DF_METH_COUNT_PHASED_DAD.filter(pl.col('start_hap_map_block').is_not_null()).head(10)

In [ ]:
DF_METH_COUNT_HAP1_HAP2_MOM = read_meth_hap1_hap2(
    pb_cpg_tool_mode='count',
    bed_hap1=MOM_METH[0],
    bed_hap2=MOM_METH[1],
)
DF_METH_COUNT_PHASED_MOM = phase_meth_to_haplotypes(DF_METH_COUNT_HAP1_HAP2_MOM, DF_HAP_MAP, 'mom')
print(f'Mom phased meth: {len(DF_METH_COUNT_PHASED_MOM)} rows')
DF_METH_COUNT_PHASED_MOM.filter(pl.col('start_hap_map_block').is_not_null()).head(10)

### Notes

- The dev data covers only a ~1Mb region of chr1, so the hap map will be small.
- Methylation phasing cells require running `aligned_bam_to_cpg_scores` on the WhatsHap-haplotagged BAMs in `dev-data/output/` first.
- The methylation BED file name pattern may need adjustment depending on how `aligned_bam_to_cpg_scores` is configured.